# 📝 Assignment — Chapter 10 — Rough, Temporal, and Fuzzy Modelling — agentic lab

This is the **student** notebook. It contains 3 graded tasks.

Work top to bottom. Where you meet a **Task**, replace the `NotImplementedError` with your implementation and run the checks cell that follows. **The assertions in the checks cells are the marking scheme** — when they all pass, the assignment is complete.

Cells outside the tasks are worked examples. Read and run them: they build what the tasks need.

> Worked solutions: [`04_agentic_lab_solution.ipynb`](04_agentic_lab_solution.ipynb)

# Chapter 10 — Rough, Temporal, and Fuzzy Modelling
### Notebook 4 · Agentic lab — choosing a formalism, and paying for search

*Book reference: Extends §10.1–10.2*

An agent whose reflex is 'model it crisply', and an MDP that makes it prove inconsistency rather than assert it.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch10_toolkit as ch10
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

In [ ]:
import ch10_agentic as AG
from oe_course import evaluation as ev, llm, mdp, optimize as opt
import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

**By the end of this notebook you can:**

1. Score a choice **and its cost**, so an agent cannot buy accuracy with unlimited expressivity.
2. Recognise the default-answer effect: a rule for behaviour the agent already has is never learned.
3. Model **constraint propagation** as a search MDP with early exit.
4. Spot an asymmetric reward that lets one verdict be claimed for free.

> **Prerequisite:** the Chapter 1 agentic lab.

## 1. Tools

`check_temporal_consistency` is the one that changes outcomes: humans — and language models — are poor at spotting cycles in interval constraints, and this is a decision procedure for exactly that.

In [ ]:
ctx = AG.Ch10Context()
tools = {t.name: t for t in AG.build_toolset(ctx)}
for name, t in tools.items():
    print(f'{name:28s} {list(t.args_schema.model_json_schema().get("properties", {}))}')
    print(f'{"":28s} {t.description.splitlines()[0]}')

In [ ]:
print(tools['relation_between'].invoke(
    {'a_start': 0, 'a_end': 3, 'b_start': 3, 'b_end': 5}))
print(tools['check_temporal_consistency'].invoke(
    {'constraints': '{"A,B": ["b"], "B,C": ["b"], "A,C": ["bi"]}'}))
print(tools['fuzzy_membership'].invoke({'set_name': 'tall', 'value': 182}))
print(tools['rough_approximation'].invoke({'target': 'p1,p3,p5'}))
print(tools['expressivity_cost'].invoke({'formalism': 'temporal'}))
print('\ntrajectory:', ctx.log.names())

## 2. The dataset

Twelve requirements, three per formalism, stratified so both halves see all four. The agent answers **which formalism** and **what it costs** — because an agent scored only on the choice can buy accuracy by always reaching for the most expressive option.

In [ ]:
train, dev = AG.build_dataset('train'), AG.build_dataset('dev')
print(pd.DataFrame([{'id': e.id, 'formalism': e.gold_formalism,
                     'cost': e.gold_cost,
                     'split': 'train' if e in train else 'dev'}
                    for e in AG.build_dataset('all')]).to_string(index=False))
assert ({e.gold_formalism for e in train} == {e.gold_formalism for e in dev})

In [ ]:
lm = llm.configure_dspy(AG.FORMALISM_RULEBOOK, AG.formalism_responder)
baseline = AG.FormalismProgram()
for e in dev:
    p = baseline(**e.inputs())
    print(f'{e.id:24s} {p.formalism:9s}/{p.cost:5s}  '
          f'gold {e.gold_formalism:9s}/{e.gold_cost}')

In [ ]:
before = ev.evaluate_dataset(baseline, dev, AG.formalism_scorer)
print('BEFORE:', before['mean_score'])
print('violations:', before['violations'])

In [ ]:
gepa_metric = ev.make_gepa_metric(AG.formalism_scorer, AG.FORMALISM_RULEBOOK)
reflect = llm.reflection_lm(AG.FORMALISM_RULEBOOK, AG.formalism_responder)
tuned = opt.run_gepa(baseline, train, gepa_metric, valset=train,
                     max_metric_calls=100, reflection_lm=reflect)
result = opt.compare(AG.FormalismProgram(), tuned, dev, AG.formalism_scorer)
print(result.report()[:1400])

In [ ]:
found = AG.FORMALISM_RULEBOOK.active_in(result.instruction_after)
print('rules discovered:', sorted(found))
print('rules not needed:', sorted(set(AG.FORMALISM_RULEBOOK.ids) - found))
print('\nThe undiscovered rule describes the agent\'s DEFAULT: it already\n'
      'reaches for crisp, so the metric never punished it and there was\n'
      'nothing to learn. As in Chapter 8, an unlearned rule is not\n'
      'automatically a failure -- check whether it was ever violated.')

## 3. Constraint propagation as an MDP

A solver narrows label sets by composing through a third interval. Each composition costs; the agent chooses **which** to do and **when to stop**.

| | |
|---|---|
| **S** | the three label sets, and whether a verdict was given |
| **A** | propagate through A, B or C; or declare consistent / inconsistent |
| **T** | deterministic — composition is a function |
| **R** | −cost per propagation; `+1` for a **justified** correct verdict |

As in Chapter 2, *justified* is load-bearing: declaring inconsistency is only rewarded once a label has actually emptied.

In [ ]:
M = AG.PropagationMDP(ab={'b'}, bc={'b'}, ac={'bi'}, consistent=False)
print('network: A before B, B before C, A after C  (inconsistent)')
print('reachable states:', len(M.states()))
V, pi = mdp.value_iteration(M)
state = M.initial_state()
print(f'V*(s0) = {V[state]:.3f}\n')
while not M.is_terminal(state):
    action = pi[state]
    print(f'  {str(state):22s} -> {action}')
    state = M.transition(state, action)[0][1]

One propagation is enough: composing through **A** empties the B–C label, which witnesses the contradiction. The agent then declares — and gets paid, because the claim is backed by an empty label rather than a hunch.

In [ ]:
rows = []
for name, kwargs in [
        ('inconsistent cycle', dict(ab={'b'}, bc={'b'}, ac={'bi'}, consistent=False)),
        ('consistent chain', dict(ab={'b'}, bc={'b'}, ac={'b'}, consistent=True)),
        ('consistent, AC open',
         dict(ab={'b'}, bc={'b'}, ac=set(AG.MDP_RELATIONS), consistent=True)),
        ('inconsistent via meets',
         dict(ab={'m'}, bc={'m'}, ac={'eq'}, consistent=False))]:
    Mx = AG.PropagationMDP(**kwargs)
    Vx, pix = mdp.value_iteration(Mx)
    st, plan = Mx.initial_state(), []
    while not Mx.is_terminal(st):
        a = pix[st]; plan.append(a.replace('propagate:', 'prop-')
                                  .replace('declare:', ''))
        st = Mx.transition(st, a)[0][1]
    rows.append({'network': name, 'states': len(Mx.states()),
                 'V*': round(Vx[Mx.initial_state()], 3), 'plan': ' -> '.join(plan)})
print(pd.DataFrame(rows).to_string(index=False))

> **Look at the asymmetry.** Inconsistency costs a propagation to witness; consistency is declared immediately, for free, at `V* = 1.0`.

That is a **reward-design flaw**, not a discovery about temporal reasoning. Path consistency is *incomplete*, so a network that survives propagation is not actually proven consistent — yet this reward pays full marks for saying so without doing any work. Exercise 4.2 fixes it.

### Task 4.1 — Make propagation expensive

Raise the propagation cost until the agent stops bothering to prove inconsistency. Report the threshold and explain it.

In [ ]:
# YOUR CODE HERE

raise NotImplementedError


**Checks for Task 4.1.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
assert [r['cost'] for r in rows] == [0.05, 0.3, 0.5, 0.9, 1.5]
assert rows[0]['propagations'] > 0, 'cheap propagation should be used'
# Raising the price of propagation cannot raise the optimal value, nor buy more
# propagations than the cheapest setting did.
assert rows[-1]['V*'] <= rows[0]['V*'] + 1e-9
assert rows[-1]['propagations'] <= rows[0]['propagations']

### Task 4.2 — Fix the asymmetric reward

Require a consistency claim to be justified too — say, by at least one propagation that changed nothing. Show the optimal policy now doing work before declaring consistency.

> **Hint.** `PropagationMDP(..., require_check=True)` makes a consistency claim require at least one propagation.

In [ ]:
# YOUR CODE HERE: PropagationMDP takes a require_check flag

raise NotImplementedError


**Checks for Task 4.2.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
assert not any(a.startswith('propagate') for a in loose)
assert any(a.startswith('propagate') for a in strict)
assert strict_value < loose_value

### Task 4.3 — Score the formalism only, and watch the agent overspend

Build a metric that ignores the cost half, optimise against it, and show the resulting agent has learned nothing about what expressivity costs.

In [ ]:
# YOUR CODE HERE

raise NotImplementedError


**Checks for Task 4.3.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
assert 'price-the-expressivity' not in blind_rules

## Chapter 10 in the course arc

| | Ch. 8 | Ch. 9 | Ch. 10 |
|---|---|---|---|
| MDP | serve under staleness | optimal stopping | **propagation with early exit** |
| second half of the score | execution strategy | presentation | **the price of the choice** |
| the failure it prevents | stale answers | unreadable output | **unaffordable reasoning** |

Chapter 10 adds a habit worth keeping: **score the cost, not just the answer**. An agent judged only on correctness will happily buy it with expressivity you cannot afford — and it will look perfect in the report while doing so.